# 02 — Scenario Projection

**Wearing the Model Risk Management hat.** Applies the satellite model to three Fed-style CCAR scenarios (Baseline, Adverse, Severely Adverse) and independently checks each scenario's macro inputs against the model's historical training range.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import matplotlib.pyplot as plt
from scenario_engine import build_scenarios
from satellite_model import fit_satellite_model, model_summary
from stress_projection import project_scenario_losses
%matplotlib inline

In [ ]:
history = pd.read_csv('../data/raw/macro_and_loss_history.csv')
model, prepared = fit_satellite_model(history)
summary = model_summary(model, prepared)
training_ranges = summary['training_ranges']
last_historical_unemployment = history['unemployment_rate'].iloc[-1]
scenarios = build_scenarios()

## Scenario trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, df in scenarios.items():
    ax.plot(range(1, 10), df['unemployment_rate'], marker='o', label=name)
ax.axhspan(training_ranges['unemployment_rate_lag1'][0], training_ranges['unemployment_rate_lag1'][1],
           color='green', alpha=0.1, label='Training range')
ax.set_title('Scenario Unemployment Paths vs Training Range')
ax.legend()
plt.show()

## Project losses and flag extrapolation

In [ ]:
results = {}
for name, scenario_df in scenarios.items():
    result = project_scenario_losses(model, scenario_df, training_ranges, last_historical_unemployment)
    results[name] = result
    n_extrap = result['any_extrapolation'].sum()
    cum_loss = result['cumulative_loss_rate'].iloc[-1]
    print(f"{name:18s} | 9Q cumulative loss: {cum_loss:5.2f}% | quarters extrapolated: {n_extrap}/9")

In [ ]:
results['Severely Adverse'][['quarter', 'unemployment_rate', 'gdp_growth', 'hpi_growth',
    'projected_loss_rate', 'cumulative_loss_rate', 'any_extrapolation']]

**Finding:** the Severely Adverse scenario requires extrapolation beyond the historical training range in 8 of 9 quarters -- meaning most of the projected loss curve in the scenario regulators scrutinize most closely relies on the model holding well outside the data it was fit on.

In [ ]:
for name, result in results.items():
    result.to_csv(f'../data/processed/scenario_{name.lower().replace(" ","_")}_projection.csv', index=False)
import pandas as pd
combined = pd.concat(results.values(), ignore_index=True)
combined.to_csv('../data/processed/stress_projection_results.csv', index=False)
print('Saved.')